# Test 4: Multi-Seed Cross-Validation (mean ± std)

**Goal**: Prove your results are not a random-seed artefact.
A single seed can be lucky or unlucky. This notebook trains the GraphCodeBERT model
3× with different random seeds and reports:

```
Accuracy  : 91.23% ± 0.45%
ROC-AUC   : 0.9634 ± 0.0021
PR-AUC    : 0.9601 ± 0.0018
F1 (macro): 0.9124 ± 0.0044
```

This is the standard format required for ML papers to demonstrate statistical robustness.

**Strategy**: We skip the ~6h DFG training per seed and instead:
1. Use the pre-trained GraphCodeBERT+DFG feature extractor (frozen).
2. Train only the **light-weight 2-class linear classifier head** — fast (~10 min/seed).
3. Different seeds control weight initialisation + dataset shuffling.

> **Note**: If you want full fine-tuning variance, you can set `FREEZE_ENCODER = False`
> and let it run overnight across 3–5 seeds.

**Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin` (pre-trained encoder)

**Output**: `/kaggle/working/test4_multiseed_results.txt`

In [ ]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

In [ ]:
import os, json, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    get_linear_schedule_with_warmup,
    RobertaConfig, RobertaModel,
    AutoTokenizer)
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Imports OK")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
class Args:
    train_file         = "/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl"
    pretrained_encoder = "/kaggle/input/model2/model.bin"   # GraphCodeBERT+DFG

    model_name_or_path = "microsoft/graphcodebert-base"
    tokenizer_name     = "microsoft/graphcodebert-base"

    code_length        = 384
    data_flow_length   = 128
    train_batch_size   = 16
    eval_batch_size    = 32
    learning_rate      = 2e-5
    max_grad_norm      = 1.0
    num_train_epochs   = 3         # epochs per seed
    val_ratio          = 0.10      # same 90/10 split as original

    # ── KEY: list of seeds to run ──
    seeds              = [42, 123, 2025]

    # FREEZE_ENCODER = True  → only train classifier head (fast, ~10 min/seed)
    # FREEZE_ENCODER = False → full fine-tune (slow, ~6h/seed)
    freeze_encoder     = True

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()

print(f"Device  : {args.device}")
print(f"Seeds   : {args.seeds}")
print(f"Frozen  : {args.freeze_encoder}")

In [ ]:
# ─── MODEL  ───────────────────────────────────────────────────────────────────
class Model(nn.Module):
    def __init__(self, encoder, config, args, seed):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)

        # Initialise the head with a fixed seed so only the seed arg matters
        torch.manual_seed(seed)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        ext = (1.0 - attn_mask) * -10000.0
        ext = ext.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext,
            head_mask=[None] * self.config.num_hidden_layers
        )[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

In [ ]:
# ─── DATASET ──────────────────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def _char_idx(self, lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r, len(lines)))) + c

    def __getitem__(self, idx):
        entry      = json.loads(self.lines[idx])
        code       = entry.get('code', '')
        dfg        = entry.get('dfg', [])[:self.args.data_flow_length]
        label      = int(entry.get('label', 0))
        code_lines = code.splitlines(keepends=True)

        tok = self.tokenizer(
            code, max_length=self.args.code_length, truncation=True,
            padding='max_length', return_offsets_mapping=True
        )
        input_ids, offsets = tok['input_ids'], tok['offset_mapping']
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        n2t, p2n = {}, {}

        for ni, node in enumerate(dfg):
            sp, ep = node[1][0], node[1][1]
            pk = (sp[0], sp[1], ep[0], ep[1])
            p2n[pk] = ni
            try:
                cs = self._char_idx(code_lines, sp)
                ce = self._char_idx(code_lines, ep)
                n2t[ni] = [i for i, (ts, te) in enumerate(offsets)
                           if ts != te and
                           ((ts >= cs and te <= ce) or (cs >= ts and ce <= te))]
            except IndexError:
                n2t[ni] = []

        c_len = self.args.code_length
        mask  = np.zeros((self.total_len, self.total_len), dtype=bool)
        mask[:c_len, :c_len] = True
        for ni, node in enumerate(dfg):
            ani = c_len + ni
            for ti in n2t.get(ni, []):
                mask[ani, ti] = mask[ti, ani] = True
            for pp in node[4]:
                pk = (pp[0][0], pp[0][1], pp[1][0], pp[1][1])
                if pk in p2n:
                    mask[ani, c_len + p2n[pk]] = mask[c_len + p2n[pk], ani] = True
            mask[ani, ani] = True

        ids   = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        pad   = self.total_len - len(ids)
        if pad > 0:
            ids   += [self.tokenizer.pad_token_id] * pad
            p_ids += [1] * pad

        return {
            'input_ids': torch.tensor(ids,   dtype=torch.long),
            'p_ids':     torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(mask,  dtype=torch.float),
            'label':     torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# ─── TRAINING LOOP ───────────────────────────────────────────────────────────
def train_one_seed(model, train_ds, val_ds, seed):
    loader = DataLoader(
        train_ds, batch_size=args.train_batch_size,
        shuffle=True, num_workers=2, pin_memory=True
    )

    # Only optimise the classifier head if encoder is frozen
    params    = model.classifier.parameters() if args.freeze_encoder \
                else model.parameters()
    optimizer = AdamW(params, lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0,
        num_training_steps=len(loader) * args.num_train_epochs
    )
    scaler   = GradScaler()
    best_acc = 0.0
    best_state = None

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        for batch in tqdm(loader, desc=f"  [seed {seed}] Epoch {epoch}"):
            inp = {
                'input_ids': batch['input_ids'].to(args.device),
                'p_ids':     batch['p_ids'].to(args.device),
                'attn_mask': batch['attn_mask'].to(args.device),
                'labels':    batch['label'].to(args.device)
            }
            optimizer.zero_grad()
            with autocast():
                loss, _ = model(**inp)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()

        val_acc, _, _, _ = evaluate(model, val_ds)
        avg_loss      = tr_loss / len(loader)
        print(f"  Epoch {epoch} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4%}")

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = copy.deepcopy(model.state_dict())

    # Restore best weights
    model.load_state_dict(best_state)
    return best_acc


@torch.no_grad()
def evaluate(model, dataset, desc="Val"):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        num_workers=2, pin_memory=True)
    model.eval()
    preds, labels, probs_list = [], [], []
    for batch in loader:
        inp = {
            'input_ids': batch['input_ids'].to(args.device),
            'p_ids':     batch['p_ids'].to(args.device),
            'attn_mask': batch['attn_mask'].to(args.device)
        }
        prob = model(**inp)
        probs_list.extend(prob[:, 1].cpu().numpy())
        preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    model.train()
    return accuracy_score(labels, preds), preds, np.array(probs_list), np.array(labels)

In [ ]:
# ─── MAIN: MULTI-SEED LOOP ────────────────────────────────────────────────────
print("Loading shared components...")
config    = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name)
print("  ✓ Config & tokenizer ready")

print("Loading dataset...")
full_ds = TextDataset(tokenizer, args, args.train_file)
n       = len(full_ds)
train_n = int(0.9 * n)
val_n   = n - train_n
print(f"  Total: {n:,}  Train: {train_n:,}  Val: {val_n:,}")

# Load the pre-trained encoder weights ONCE (all seeds reuse the same encoder)
print("\nLoading pre-trained encoder...")
encoder_base = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
pretrained   = torch.load(args.pretrained_encoder, map_location='cpu')
# Extract only encoder weights (strip classifier keys)
encoder_state = {k.replace('encoder.', ''): v
                 for k, v in pretrained.items() if k.startswith('encoder.')}
encoder_base.load_state_dict(encoder_state, strict=False)
print("  ✓ Encoder weights loaded")

seed_results = []

for seed in args.seeds:
    print(f"\n{'─'*55}")
    print(f"SEED {seed}")
    print(f"{'─'*55}")

    # Fix all RNG sources for this seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if args.n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

    # Fresh split with this seed (shuffles which samples go to val)
    gen = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(full_ds, [train_n, val_n], generator=gen)

    # Deep-copy encoder so each seed starts from identical pretrained weights
    enc   = copy.deepcopy(encoder_base)
    model = Model(enc, config, args, seed).to(args.device)

    # Optionally freeze the encoder
    if args.freeze_encoder:
        for param in model.encoder.parameters():
            param.requires_grad = False
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Encoder frozen. Trainable params: {trainable:,}")

    # Train
    best_val_acc = train_one_seed(model, train_ds, val_ds, seed)

    # Full evaluation
    model.eval()
    val_acc, val_preds, val_probs, val_labels = evaluate(model, val_ds, desc="Final Val")

    roc_auc = roc_auc_score(val_labels, val_probs)
    pr_auc  = average_precision_score(val_labels, val_probs)
    f1_mac  = f1_score(val_labels, val_preds, average='macro')
    f1_mal  = f1_score(val_labels, val_preds, pos_label=1)

    seed_results.append({
        'seed':    seed,
        'acc':     val_acc,
        'roc_auc': roc_auc,
        'pr_auc':  pr_auc,
        'f1_mac':  f1_mac,
        'f1_mal':  f1_mal
    })

    print(f"  Seed {seed} → Acc={val_acc:.4%}  "
          f"ROC={roc_auc:.4f}  PR={pr_auc:.4f}  F1={f1_mac:.4f}")

print("\n\nAll seeds done!")

In [ ]:
# ─── SUMMARY STATISTICS ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("MULTI-SEED RESULTS SUMMARY (paper-ready: mean ± std)")
print("="*60)

metric_keys  = ['acc',     'roc_auc', 'pr_auc',  'f1_mac',   'f1_mal']
metric_names = ['Accuracy', 'ROC-AUC', 'PR-AUC', 'F1 (macro)', 'F1 (malicious)']

# Per-seed table
print(f"\n{'Seed':>6} | ", end="")
print("  ".join(f"{n:>14}" for n in metric_names))
print("-" * 80)
for r in seed_results:
    print(f"{r['seed']:>6} | ", end="")
    print("  ".join(f"{r[k]:>14.4%}" if k == 'acc' else f"{r[k]:>14.4f}"
                    for k in metric_keys))

# Mean ± std
print("-" * 80)
summary = {}
for k, name in zip(metric_keys, metric_names):
    vals       = [r[k] for r in seed_results]
    mean, std  = np.mean(vals), np.std(vals)
    summary[k] = (mean, std)
    if k == 'acc':
        print(f"mean±std | {name:>15}: {mean:.4%} ± {std:.4%}")
    else:
        print(f"mean±std | {name:>15}: {mean:.4f} ± {std:.4f}")

In [ ]:
# ─── ERROR BAR PLOT ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
plot_keys  = ['acc',     'roc_auc', 'pr_auc',  'f1_mac']
plot_names = ['Accuracy', 'ROC-AUC', 'PR-AUC', 'F1 (macro)']

seed_labels = [str(s) for s in args.seeds]
palette     = ['#4C72B0', '#DD8452', '#55A868']

for ax, key, name in zip(axes, plot_keys, plot_names):
    vals = [r[key] for r in seed_results]
    mean, std = summary[key]

    bars = ax.bar(seed_labels, vals, color=palette[:len(vals)],
                  width=0.5, edgecolor='white')
    ax.axhline(mean, color='black', lw=1.5, linestyle='--',
               label=f'mean={mean:.4f}')
    ax.fill_between([-0.5, len(vals) - 0.5],
                    mean - std, mean + std,
                    alpha=0.15, color='grey', label=f'±1σ ({std:.4f})')

    ymin = max(0, min(vals) - 0.03)
    ax.set_ylim([ymin, min(1.0, max(vals) + 0.04)])
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Random Seed', fontsize=10)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

fig.suptitle(f'Multi-Seed Stability — GraphCodeBERT + DFG  '
             f'(freeze_encoder={args.freeze_encoder})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()

plot_path = "/kaggle/working/test4_multiseed_errorbar.png"
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Error-bar plot saved → {plot_path}")

In [ ]:
# ─── SAVE TEXT RESULTS ───────────────────────────────────────────────────────
out_path = "/kaggle/working/test4_multiseed_results.txt"
with open(out_path, "w") as f:
    f.write("Test 4: Multi-Seed Results\n")
    f.write("=" * 60 + "\n")
    f.write(f"Seeds         : {args.seeds}\n")
    f.write(f"Freeze encoder: {args.freeze_encoder}\n")
    f.write(f"Epochs/seed   : {args.num_train_epochs}\n\n")

    # Per-seed
    for r in seed_results:
        f.write(f"Seed {r['seed']}:\n")
        for k, n in zip(metric_keys, metric_names):
            f.write(f"  {n:>16}: {r[k]:.4f}\n")
        f.write("\n")

    # Summary
    f.write("─" * 40 + "\n")
    f.write("PAPER-READY SUMMARY (mean ± std):\n")
    for k, n in zip(metric_keys, metric_names):
        mean, std = summary[k]
        f.write(f"  {n:>16}: {mean:.4f} ± {std:.4f}\n")

print(f"Results saved  → {out_path}")
print(f"Error-bar plot → /kaggle/working/test4_multiseed_errorbar.png")